# Re-training unsupervised SimCSE
COMP8240 — Trilok Kumar Reddy Ande

The official `train.py` was written for `transformers` 4.2.1 and depends on internal parts of the library that have since changed. This notebook is a **minimal re-implementation of the paper's unsupervised objective**, using the paper's settings for BERT-base:

* data: the authors' 1M English Wikipedia sentences
* batch size 64, learning rate 3e-5, max length 32, temperature 0.05, 1 epoch
* positive pair = the same sentence passed twice through BERT (different dropout masks)
* an MLP layer on top of [CLS] used only during training; plain [CLS] at test time

Difference from the paper: the paper checks the STS-B development set every 125 steps and keeps the best checkpoint. This notebook simply keeps the final model.

Run on a **T4 GPU**: Runtime → Run all. Training on 1M sentences takes roughly 30–60 minutes.

## 1. Get the SimCSE evaluation code and STS data

In [1]:
%cd /content
!rm -rf SimCSE
!git clone -q https://github.com/princeton-nlp/SimCSE.git
%cd /content/SimCSE
!pip install -q transformers scipy scikit-learn prettytable
!sed -i "s/tokenizer.batch_encode_plus(/tokenizer(/g" evaluation.py
%cd SentEval/data/downstream/
!bash download_dataset.sh > /dev/null 2>&1
%cd /content

/content
/content/SimCSE
/content/SimCSE/SentEval/data/downstream
/content


## 2. Download the authors' 1M Wikipedia training sentences

In [2]:
!wget -q -O /content/wiki1m.txt https://huggingface.co/datasets/princeton-nlp/datasets-for-simcse/resolve/main/wiki1m_for_simcse.txt
!wc -l /content/wiki1m.txt

1000000 /content/wiki1m.txt


## 3. Train

In [3]:
import torch, random, time, sys, transformers
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0))

N_TRAIN = 1_000_000                     # full paper size. Use 100_000 for a quick (~5 min) test.
BATCH, LR, MAXLEN, TEMP = 64, 3e-5, 32, 0.05   # paper's settings for unsupervised BERT-base
random.seed(42); torch.manual_seed(42)

lines = [l.strip() for l in open("/content/wiki1m.txt", encoding="utf-8") if l.strip()]
random.shuffle(lines)
train = lines[:N_TRAIN]
print("Training sentences:", len(train))

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
enc = AutoModel.from_pretrained("bert-base-uncased").cuda().train()   # dropout is ON: this is the "augmentation"
mlp = nn.Sequential(nn.Linear(768, 768), nn.Tanh()).cuda()           # MLP used only during training
opt = torch.optim.AdamW(list(enc.parameters()) + list(mlp.parameters()), lr=LR)
steps = len(train) // BATCH
sched = get_linear_schedule_with_warmup(opt, 0, steps)
scaler = torch.amp.GradScaler()

t0 = time.time()
for step in range(steps):
    x = tok(train[step * BATCH:(step + 1) * BATCH], padding=True, truncation=True,
            max_length=MAXLEN, return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        z1 = mlp(enc(**x).last_hidden_state[:, 0])   # same batch twice ...
        z2 = mlp(enc(**x).last_hidden_state[:, 0])   # ... with different dropout masks
    sim = F.cosine_similarity(z1.float().unsqueeze(1), z2.float().unsqueeze(0), dim=-1) / TEMP
    loss = F.cross_entropy(sim, torch.arange(sim.size(0), device="cuda"))  # positives on the diagonal
    opt.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    sched.step()
    if step % 1000 == 0:
        print(f"step {step}/{steps}  loss {loss.item():.4f}  {time.time() - t0:.0f}s")

enc.save_pretrained("/content/my-unsup-simcse")
tok.save_pretrained("/content/my-unsup-simcse")
print("Training finished in", round((time.time() - t0) / 60, 1), "minutes")

torch 2.11.0+cu130 | transformers 5.22.0 | GPU Tesla T4
Training sentences: 1000000


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

... (2 outputs left out)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_478/569921673.py:36: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will

step 0/15625  loss 1.1290  1s
step 1000/15625  loss 0.0001  209s
step 2000/15625  loss 0.0000  417s
step 3000/15625  loss 0.0001  625s
step 4000/15625  loss 0.0000  833s
step 5000/15625  loss 0.0000  1041s
step 6000/15625  loss 0.0001  1249s
step 7000/15625  loss 0.0000  1457s
step 8000/15625  loss 0.0000  1665s
step 9000/15625  loss 0.0000  1873s
step 10000/15625  loss 0.0000  2082s
step 11000/15625  loss 0.0003  2290s
step 12000/15625  loss 0.0000  2499s
step 13000/15625  loss 0.0000  2707s
step 14000/15625  loss 0.0000  2915s
step 15000/15625  loss 0.0000  3124s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished in 54.3 minutes


## 4. Evaluate our re-trained model with the official script

In [4]:
%cd /content/SimCSE
!python evaluation.py --model_name_or_path /content/my-unsup-simcse --pooler cls_before_pooler --task_set sts --mode test 2>&1 | tail -n 12

/content/SimCSE

------ test ------
+-------+-------+-------+-------+-------+--------------+-----------------+-------+
| STS12 | STS13 | STS14 | STS15 | STS16 | STSBenchmark | SICKRelatedness |  Avg. |
+-------+-------+-------+-------+-------+--------------+-----------------+-------+
| 63.31 | 78.24 | 70.33 | 77.07 | 78.06 |    71.66     |      69.15      | 72.55 |
+-------+-------+-------+-------+-------+--------------+-----------------+-------+
+------+------+------+------+------+------+------+------+
|  MR  |  CR  | SUBJ | MPQA | SST2 | TREC | MRPC | Avg. |
+------+------+------+------+------+------+------+------+
| 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 |
+------+------+------+------+------+------+------+------+


For comparison, the paper reports an average of **76.25** for unsupervised SimCSE (BERT-base).